# 02 — Non-Commuting States, Same Diagonal

In `01_diagonal_collapse` you saw two *different* states share a $Z$-diagonal; here you meet a sharper version of the same trap — two states that do not even commute, yet still collapse onto the identical diagonal.

**Prerequisites:** `04/01_diagonal_collapse`.

**What you'll be able to do**
- Build the equator pair $|+_x\rangle\langle+_x|$ and $|+_y\rangle\langle+_y|$ and confirm they share the $Z$-diagonal $(\tfrac12, \tfrac12)$.
- Read the **commutator's Frobenius norm** (`commutativity_norm`) as a *quantitative* gauge of the quantum-ness the diagonal discards — strictly positive here, exactly zero for commuting (classical) states.
- Show that the $Z$-diagonal is **basis-dependent**: rotate the measurement basis and the difference reappears, while classical OT — pinned to the fixed $Z$-diagonal — stays blind.

You arrive with the diagonal-collapse problem already named. In 04/01 the two states were $|+\rangle\langle+|$ and $I/2$ — a *pure* state and a *mixed* one. That contrast was real, but it was also a gentle one: those two states happen to **commute**, so a single basis diagonalises both. This notebook removes that comfort. The obstruction, it turns out, is not limited to pure-versus-mixed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.quantum.density import bloch_vector, density_matrix
from qot_course.quantum.states import KET_PLUS
from qot_course.quantum_ot.preview import (
    commutativity_norm,
    commutator,
    diagonal_in_computational_basis,
    same_diagonal,
)

np.random.seed(42)
viz.use_course_style()

## 1. A sharper question — can states that don't even *commute* share a diagonal?

In 04/01 the diagonal hid the difference between a pure superposition and a classical mixture. The two states there shared more than a diagonal: they shared an *eigenbasis*. $I/2$ is proportional to the identity, so it commutes with every operator — in particular with $|+\rangle\langle+|$. Whenever two states commute, a single basis diagonalises both at once, and in *that* basis each one genuinely is a probability vector. The collapse you saw was the loss of off-diagonal coherence within one shared frame.

Now raise the stakes. Two operators **commute** when $[\rho, \sigma] = \rho\sigma - \sigma\rho = 0$; they fail to commute when that difference is non-zero. Non-commuting states have *no* shared eigenbasis — there is no single frame in which both look like classical probability vectors at the same time. So the question sharpens:

> Can two states that do not even share a description — states with $[\rho, \sigma] \neq 0$ > — still collapse onto the *same* $Z$-diagonal?

If yes, the diagonal-collapse problem is deeper than pure-versus-mixed: it reaches states that are quantum-incompatible at the operator level, and classical OT will be blind to that incompatibility too. The smallest possible witness is a single qubit, and it lives on the equator of the Bloch sphere.

## 2. The equator pair — $|+_x\rangle\langle+_x|$ versus $|+_y\rangle\langle+_y|$

Recall the Bloch picture: a pure qubit is a unit vector $\vec r = (\langle X\rangle, \langle Y\rangle, \langle Z\rangle)$ on the sphere, and the density matrix is $\rho = \tfrac12\,(I + r_x X + r_y Y + r_z Z)$. The $Z$-diagonal of $\rho$ is $(\tfrac12(1 + r_z),\ \tfrac12(1 - r_z))$ — it depends on $r_z$ **alone**. Every state on the equator has $r_z = 0$, so every equator state carries the identical $Z$-diagonal $(\tfrac12, \tfrac12)$, no matter where it sits around the equator.

Pick two equator states pointing along orthogonal axes:

- $|+_x\rangle = \tfrac{1}{\sqrt2}(|0\rangle + |1\rangle)$ — Bloch $(1, 0, 0)$, with $\rho_x = \begin{pmatrix} \tfrac12 & \tfrac12 \\ \tfrac12 & \tfrac12 \end{pmatrix}$. Its coherence sits in the **real** off-diagonal.
- $|+_y\rangle = \tfrac{1}{\sqrt2}(|0\rangle + i\,|1\rangle)$ — Bloch $(0, 1, 0)$, with $\rho_y = \begin{pmatrix} \tfrac12 & -\tfrac{i}{2} \\ \tfrac{i}{2} & \tfrac12 \end{pmatrix}$. Its coherence sits in the **imaginary** off-diagonal.

Both are pure (entropy $0$). Both share the diagonal $(\tfrac12, \tfrac12)$. But they point along *different* Bloch axes, so they cannot be the same state — and, as we will measure, they do not commute. We build $|+_y\rangle$ directly from its amplitudes.

In [ ]:
# Two pure states on the Bloch equator, along orthogonal axes.
ket_plus_x = KET_PLUS                                   # |+_x> = (|0> + |1>)/sqrt(2)
ket_plus_y = np.array([1.0, 1.0j], dtype=complex) / np.sqrt(2)  # |+_y> = (|0> + i|1>)/sqrt(2)

rho_x = density_matrix(ket_plus_x)   # Bloch (1, 0, 0) — coherence in the real off-diagonal
rho_y = density_matrix(ket_plus_y)   # Bloch (0, 1, 0) — coherence in the imaginary off-diagonal

print('rho_x  =  |+_x><+_x|')
print(np.round(rho_x, 3))
print()
print('rho_y  =  |+_y><+_y|')
print(np.round(rho_y, 3))
print()
print(f'Bloch vector of rho_x : {np.round(bloch_vector(rho_x), 3).tolist()}')
print(f'Bloch vector of rho_y : {np.round(bloch_vector(rho_y), 3).tolist()}')

**Read the output.** The two density matrices agree perfectly on the diagonal — both read $0.5$ in the top-left and bottom-right — and differ entirely off it: $\rho_x$ carries a real $\tfrac12$ in the corners, while $\rho_y$ carries a purely imaginary $\pm\tfrac{i}{2}$. The Bloch vectors confirm the geometry: $(1, 0, 0)$ versus $(0, 1, 0)$, two unit vectors at right angles on the equator. Same shadow on the $Z$-axis, orthogonal directions in space.

### Identical $Z$-diagonals

Extract each diagonal with `diagonal_in_computational_basis` — the naive classical reduction from 04/01 — and confirm with `same_diagonal` that they coincide.

In [ ]:
diag_x = diagonal_in_computational_basis(rho_x)
diag_y = diagonal_in_computational_basis(rho_y)

# np.round(..., 12) suppresses the ~1e-16 floating-point noise from the sqrt(2) normalisation
print(f'Z-diagonal of |+_x><+_x| : {np.round(diag_x, 12)}')
print(f'Z-diagonal of |+_y><+_y| : {np.round(diag_y, 12)}')
print(f'same Z-diagonal?           {same_diagonal(rho_x, rho_y)}  (tolerance 1e-9)')

**Read the output.** Both diagonals are $(\tfrac12, \tfrac12)$ and `same_diagonal` returns `True` (the 1e-9 tolerance absorbs the floating-point rounding you would otherwise see in the last digit). So the naive reduction hands classical OT the *very same* probability vector for $\rho_x$ and $\rho_y$ — exactly as it did for $|+\rangle\langle+|$ and $I/2$ in 04/01. By the diagonal alone, these two states are indistinguishable.

### The commutator measures what the diagonal throws away

Here is the new ingredient. The diagonal being shared is only half the story; the other half is *how incompatible* the two states are as operators. The commutator $[\rho_x, \rho_y]$ captures that, and its **Frobenius norm** $\lVert [\rho_x, \rho_y] \rVert_F$ (`commutativity_norm`) turns it into a single non-negative number:

- It is **exactly zero** when the two states commute — when a shared eigenbasis exists and the pair is, in the relevant sense, *classical*.
- It is **strictly positive** when they do not commute — a quantitative gauge of the quantum-ness that no choice of basis can remove, and that the diagonal silently discards.

We compute it for the equator pair, and for contrast we compute it for two genuinely classical states: a pair of diagonal density matrices, $\rho_p = \mathrm{diag}(0.7, 0.3)$ and $\rho_q = \mathrm{diag}(0.2, 0.8)$. Diagonal matrices always commute, so their commutator norm must vanish.

In [ ]:
# The non-commuting equator pair.
comm_xy = commutator(rho_x, rho_y)
norm_xy = commutativity_norm(rho_x, rho_y)

print('commutator [rho_x, rho_y]:')
print(np.round(comm_xy, 3))
print(f'Frobenius norm ||[rho_x, rho_y]||_F = {norm_xy:.4f}   (> 0  =>  no shared eigenbasis)')
print()

# Classical contrast: two diagonal (hence commuting) states.
rho_p = np.diag([0.7, 0.3]).astype(complex)
rho_q = np.diag([0.2, 0.8]).astype(complex)
norm_pq = commutativity_norm(rho_p, rho_q)
print(f'two diagonal states  diag(0.7, 0.3), diag(0.2, 0.8):')
print(f'Frobenius norm ||[rho_p, rho_q]||_F = {norm_pq:.4f}   (= 0  =>  they commute, classical)')

**Read the output.** The equator pair has commutator $[\rho_x, \rho_y] = \mathrm{diag}(\tfrac{i}{2}, -\tfrac{i}{2})$, a purely imaginary, non-zero operator with Frobenius norm $\tfrac{1}{\sqrt2} \approx 0.7071$. That strictly positive number is the diagonal-collapse problem made *quantitative*: it is the amount of operator-level incompatibility between $\rho_x$ and $\rho_y$ that the shared diagonal $(\tfrac12, \tfrac12)$ erases. The two diagonal states, by contrast, return exactly $0$ — commuting states carry no such incompatibility, and on commuting states the classical reduction loses nothing. The commutator norm is, in effect, a dial reading zero on the classical world and lighting up the moment a pair becomes genuinely quantum.

### See it — the coherence lives in different parts of the matrix

Plot the two density matrices. Each call to `viz.plot_density_matrix` produces one figure with two panels — the real part on the left, the imaginary part on the right. We draw $\rho_x$ first, then $\rho_y$.

In [ ]:
qubit_basis = ["|0>", "|1>"]
viz.plot_density_matrix(rho_x, title=r'$|+_x\rangle\langle+_x|$  —  Bloch $(1,0,0)$', basis_labels=qubit_basis)
plt.show()

viz.plot_density_matrix(rho_y, title=r'$|+_y\rangle\langle+_y|$  —  Bloch $(0,1,0)$', basis_labels=qubit_basis)
plt.show()

**Read the figures.** Look first at the two **diagonals** (top-left and bottom-right cells). In both figures, and in both the real and imaginary panels, the diagonal reads $0.50$ on the real side and $0.00$ on the imaginary side — the matching populations classical OT saw. Now look **off the diagonal**, where the states part ways. For $\rho_x$ (first figure) the coherence is a real $0.50$ in the corners of the *real* panel, and the imaginary panel is flat zero. For $\rho_y$ (second figure) it is the reverse: the real off-diagonals are zero, and the coherence appears as $\pm 0.50$ in the *imaginary* panel. Same diagonal, coherence stored in orthogonal places — real for $\rho_x$, imaginary for $\rho_y$. That phase difference is exactly what makes the commutator non-zero, and exactly what the $Z$-diagonal cannot record.

### See it on the sphere — orthogonal directions, same shadow

The Bloch sphere makes the geometry vivid. We plot each state as a vector; $|+_x\rangle$ points along $+x$, $|+_y\rangle$ along $+y$.

In [ ]:
viz.plot_bloch(ket_plus_x, title='$|+_x\\rangle$  —  Bloch $(1, 0, 0)$')
plt.show()

viz.plot_bloch(ket_plus_y, title='$|+_y\\rangle$  —  Bloch $(0, 1, 0)$')
plt.show()

**Read the figures.** Two arrows of equal length, both lying flat on the equator (no vertical $z$-component), pointing along perpendicular axes — $+x$ first, then $+y$. The shared $Z$-diagonal is precisely their shared *vertical shadow*: project either arrow onto the $z$-axis and you get the same point, the equator's height of zero, which is the diagonal $(\tfrac12, \tfrac12)$. Classical OT, reading only that vertical shadow, sees one point for both. The right-angle between the arrows — the whole equatorial separation — is the information it discards, and the non-zero commutator is its operator-level signature.

## 3. The diagonal is basis-dependent — rotate, and the difference returns

The deepest point is that the $Z$-diagonal is not an intrinsic property of a state at all: it is what *one particular measurement* happens to record. Choose a different measurement basis and you read a different diagonal. The information distinguishing $\rho_x$ from $\rho_y$ was never destroyed — it was merely invisible to the $Z$ measurement. Rotate to the $X$ basis and it reappears at once.

Measuring in the $X$ basis means expressing $\rho$ in the $\{|+_x\rangle, |-_x\rangle\}$ frame and reading *its* diagonal. The change of basis is the Hadamard $H = \tfrac{1}{\sqrt2}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$, and the rotated state is $H\rho H^\dagger$.

In [ ]:
# Hadamard sends the X-measurement basis to the computational basis: read the X-basis
# populations as the ordinary diagonal of H @ rho @ H^dagger.
H = np.array([[1.0, 1.0], [1.0, -1.0]], dtype=complex) / np.sqrt(2)

def x_basis_diagonal(rho: np.ndarray) -> np.ndarray:
    """Populations of rho in the X basis: diagonal of H rho H^dagger (rounded for display)."""
    rotated = H @ rho @ H.conj().T
    return np.round(diagonal_in_computational_basis(rotated), 12) + 0.0  # +0.0 clears -0.0

print('In the Z basis (what classical OT reads):')
print(f'  diag_Z(rho_x) = {np.round(diag_x, 12)}')
print(f'  diag_Z(rho_y) = {np.round(diag_y, 12)}')
print('  -> identical: classical OT sees one probability vector for both.')
print()
print('In the X basis (rotate the measurement):')
print(f'  diag_X(rho_x) = {x_basis_diagonal(rho_x)}')
print(f'  diag_X(rho_y) = {x_basis_diagonal(rho_y)}')
print('  -> now they differ: the hidden difference reappears.')

**Read the output.** In the $Z$ basis both states read $(\tfrac12, \tfrac12)$ — the collapse. Rotate to the $X$ basis and the picture splits cleanly: $\rho_x = |+_x\rangle\langle+_x|$ becomes $(1, 0)$ — it *is* the $+x$ outcome with certainty — while $\rho_y$ stays at $(\tfrac12, \tfrac12)$, since $|+_y\rangle$ is an equal superposition of $|+_x\rangle$ and $|-_x\rangle$. The two states the $Z$-diagonal called identical are perfectly distinguishable the moment we turn the measurement. The difference was always there; only the basis hid it.

This is the crux. A genuine state-level metric does not depend on a chosen basis. You have already met such metrics — the Bures distance from modules 02–03, and the quantum relative entropy from 02/09 — and the quantum optimal transport this module builds joins them. Classical OT, by contrast, is welded to the fixed $Z$-diagonal: it cannot rotate, so it cannot recover what that one shadow leaves out. The non-zero commutator you measured in §2 is the basis-independent certificate that there is something to recover.

## Your turn

**Warm-up.** Add a third equator state and confirm it joins the club. Build $|+_{xy}\rangle = \tfrac{1}{\sqrt2}\big(|0\rangle + e^{i\pi/4}|1\rangle\big)$ — Bloch vector $(\tfrac{1}{\sqrt2}, \tfrac{1}{\sqrt2}, 0)$, halfway between $|+_x\rangle$ and $|+_y\rangle$ on the equator — with `density_matrix`, then print its $Z$-diagonal and check `same_diagonal` against $\rho_x$. *Predict, before running:* what diagonal must any equator state have, and why?

**Go further.** Compute `commutativity_norm` of your third state with *each* of $\rho_x$ and $\rho_y$. Is the norm zero with either one? Use the answer to argue whether *any* of the three states could be told apart by classical OT acting on their $Z$-diagonals — and connect the positive commutator norms to that verdict.

**Challenge.** The commutator norm should depend on the *angle* between two equator states. Build a family $|+_\phi\rangle = \tfrac{1}{\sqrt2}\big(|0\rangle + e^{i\phi}|1\rangle\big)$ for $\phi$ sweeping $[0, \pi]$ (each shares the diagonal $(\tfrac12, \tfrac12)$), compute `commutativity_norm` between $|+_0\rangle = |+_x\rangle$ and $|+_\phi\rangle$ at each $\phi$, and plot it against $\phi$. At which angle is the incompatibility largest, and where does it vanish? Relate the vanishing points to states that commute. (You will reuse `commutativity_norm` and `density_matrix`, already imported above, and `matplotlib.pyplot` as `plt`.)

## What you built

- You built the equator pair $|+_x\rangle\langle+_x|$ and $|+_y\rangle\langle+_y|$ — two **non-commuting** pure states — and confirmed with `same_diagonal` that they carry the identical $Z$-diagonal $(\tfrac12, \tfrac12)$.
- You read the commutator's Frobenius norm as a *quantitative* gauge of quantum-ness: $\approx 0.7071$ for the non-commuting pair, **exactly $0$** for two commuting diagonal states — a dial that lights up precisely when a pair becomes genuinely quantum.
- You saw, in the density matrices and on the Bloch sphere, that the coherence hides in *different* parts of the matrix (real versus imaginary), and you watched the hidden difference **reappear** the instant you rotated the measurement basis from $Z$ to $X$.

This is a real step beyond 04/01. The diagonal-collapse problem is not a quirk of comparing a pure state to a mixed one — it reaches states that are *incompatible at the operator level*, and you now have a number, the commutator norm, that measures exactly how much classical OT throws away. You also have the reason *why* it throws it away: the diagonal is one basis-dependent shadow, and classical OT cannot turn its head.

## What's next

In `03_trevisan_taxonomy` these observations harden into a formal principle, and we place the competing formulations of quantum optimal transport on Trevisan's map. The commutator norm you computed here is what earns its place there.

## References

- D. Trevisan, "Optimal transport methods for quantum systems", arXiv:2202.02091 (2022). DOI:10.48550/arXiv.2202.02091
- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, Cambridge University Press (2010), ch. 2 — the Bloch sphere, density matrices, and operator commutators. DOI:10.1017/CBO9780511976667

**Previous:** `notebooks/04_QuantumOptimalTransport/01_diagonal_collapse.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/03_trevisan_taxonomy.ipynb`